# The hour before, the first spike and the close back

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Interest-Rate-Decision/notebooks/blob/main/03-hour-before-range.ipynb)

This is [InterestRateDecision.com's own measurement](https://interestratedecision.com/about/?via=notebooks#method), rule for rule (the code in `ird_events.reaction_of`
gives the site's figures on the site's candles). The rules were set before any decision was counted:

| | |
|---|---|
| **the hour before** | the high and low of the 60 minutes before the announcement: the box |
| **the first side** | the edge of the box price trades through first, within 30 minutes |
| **the close back** | a five-minute candle (counted from the announcement) closing back inside the box within 60 minutes |
| **the other side** | after the close back, price trading through the opposite edge within four hours |

The read: the decision brings a burst of orders within a minute; the first spike runs the stops resting beyond one side
of the hour before (a stop hunt), and when it closes back inside, the move often belongs to the other side.
The counts below include every decision where the first spike did **not** close back: they say how often it happened,
and how often it did not.

### Candles: download your own

Dukascopy's candles are not ours to share, so this repository ships none (only a synthetic sample). Two ways to get them:

1. **The `dukascopy-node` CLI** (needs Node.js): `python scripts/download_candles.py --banks fed ecb --since 2020-01-01` runs
   `npx dukascopy-node -i eurusd -from ... -to ... -t m1 -f csv` for the hours around each decision and writes `candles/EURUSD/`.
   Add `--dry-run` to see the commands first.
2. **By hand** from Dukascopy's historical data export (one-minute, bid, GMT): save each CSV in `candles/<PAIR>/`.

`ird.read_candle_csv` reads both formats. Then set `SAMPLE = False` below.


In [ ]:
# On Colab or Kaggle, fetch the repository first: the helper module and the synthetic sample.
import os, subprocess, sys
if not os.path.exists("ird_events.py"):
    if not os.path.exists("notebooks-repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Interest-Rate-Decision/notebooks.git", "notebooks-repo"], check=True)
    os.chdir("notebooks-repo")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ird_events as ird

In [ ]:
SAMPLE = True                    # True: synthetic candles from data/sample, offline. False: your own downloads.
CANDLES_DIR = "candles"          # scripts/download_candles.py writes candles/<PAIR>/*.csv
BANKS = ["fed", "ecb", "boe"]    # any of: fed ecb boe boj boc rba rbnz snb norges riksbank
SINCE = "2015-01-01"             # IRD's decisions go back to 2008

ird.banner(SAMPLE)
decisions = ird.load_decisions(sample=SAMPLE)
decisions = decisions[decisions.bank.isin(BANKS) & (decisions.time_utc >= pd.Timestamp(SINCE, tz="UTC"))]
candles = {
    pair: ird.load_sample_candles(pair) if SAMPLE else ird.load_candles(pair, CANDLES_DIR)
    for pair in decisions.pair.dropna().unique()
}
print(len(decisions), "decisions;", {pair: len(c) for pair, c in candles.items()}, "one-minute candles")

In [ ]:
rows, measured = [], {}
for d in decisions.itertuples():
    r = ird.reaction_of(candles[d.pair], d.time_utc, d.pair)
    if r is None:
        continue  # fewer than 20 candles in the hour before, or none after
    measured[(d.bank, d.date)] = r
    rows.append({"bank": d.bank, "date": d.date, "move": d.move, "box_pips": round(r.box_pips, 1),
                 "first": r.first, "first_at": r.first_at, "first_pips": None if r.first_pips is None else round(r.first_pips, 1),
                 "closed_back": r.closed_back, "closed_back_at": r.closed_back_at, "other_side": r.other_side})
table = pd.DataFrame(rows)
table.tail(10)

## How often, by bank

In [ ]:
def counts(part):
    n = len(part)
    broke = part["first"].notna().sum()
    back = (part["closed_back"] == True).sum()
    other = (part["other_side"] == True).sum()
    return pd.Series({
        "decisions": n,
        "broke an edge": broke,
        "closed back": back,
        "closed back, % of broke": round(100 * back / broke, 1) if broke else np.nan,
        "reached the other side": other,
        "other side, % of closed back": round(100 * other / back, 1) if back else np.nan,
        "median hour-before range, pips": part["box_pips"].median(),
    })

tallies = table.groupby("bank").apply(counts, include_groups=False)
whole = ["decisions", "broke an edge", "closed back", "reached the other side"]
tallies.astype({c: int for c in whole})

## One decision, drawn

The hour before as a band, the announcement as a line, and the close back when there was one. Change `PICK` to look at another.

In [ ]:
PICK = -1  # the latest; or an index into table
row = table.iloc[PICK]
r = measured[(row.bank, row.date)]
part = ird.window_of(candles[r.pair], r.t0, before=90, after=240)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.vlines(part.minute, part.low, part.high, color="tab:gray", lw=0.8)
ax.plot(part.minute, part.close, color="black", lw=0.8)
ax.axhspan(r.box_low, r.box_high, xmin=0, xmax=1, color="tab:orange", alpha=0.15, label="the hour before")
ax.axvline(0, color="tab:red", lw=1, label="announcement")
if r.closed_back_at:
    ax.axvline(r.closed_back_at, color="tab:green", lw=1, ls="--", label=f"the close back (+{r.closed_back_at} min)")
ax.set_title(f"{ird.BANKS[row.bank]['name']}, {row.date}: first side {r.first or 'none'}, "
             f"closed back {r.closed_back}, other side {r.other_side}")
ax.set_xlabel("minutes from the announcement")
ax.legend(loc="upper left")
plt.show()

## By move

Does the close back happen more often on holds than on hikes and cuts?

In [ ]:
broke = table[table["first"].notna()]
(broke.assign(closed_back=broke.closed_back == True, other_side=broke.other_side == True)
      .groupby(["bank", "move"])[["closed_back", "other_side"]]
      .agg(["mean", "count"]).round(2))

## Reading it

- The close back is a count, not a promise: the share that did not close back is in the same table.
- The hour before is often narrow, so a first spike through it is almost certain; what differs between banks is how often the close back follows, and how often the other side is reached after it.
- The site publishes the same counts for every bank with the latest candles, and draws each decision: start at [the Fed's page](https://interestratedecision.com/fed/?via=notebooks) or [how to trade FOMC](https://interestratedecision.com/blog/how-to-trade-fomc/?via=notebooks).

---

Decisions from [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) (free to reuse with a link). The method: [how the site measures a reaction](https://interestratedecision.com/about/?via=notebooks#method). The studies: [how often central banks surprise](https://interestratedecision.com/blog/how-often-do-central-banks-surprise/?via=notebooks), [the most volatile forex news](https://interestratedecision.com/blog/most-volatile-forex-news/?via=notebooks), [how to trade FOMC](https://interestratedecision.com/blog/how-to-trade-fomc/?via=notebooks), [can central bank decisions be predicted?](https://interestratedecision.com/blog/can-central-bank-decisions-be-predicted/?via=notebooks). Learn to test and trade the news: [Forexia's courses](https://forexiapro.com/courses?via=notebooks).

Not financial advice. Past reactions say nothing certain about the next one.

Part of the Forexia network: [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) · [Forexia Academy](https://forexiapro.com/courses?via=notebooks) · [TradingSignalBot](https://tradingsignalbot.com/?via=notebooks) · [PropSim](https://github.com/PropSim-Enterprise) · [OpenCharts](https://github.com/dylanpersonguy/OpenCharts)